In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score

# 1. Dictionnaire des modèles à comparer
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []

# 2. Entraînement et Évaluation rapide
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4)
    })

# 3. Affichage du Tableau Comparatif
import pandas as pd
df_results = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
print("=== BENCHMARK DES MODÈLES ML ===")
print(df_results.to_string(index=False))

=== BENCHMARK DES MODÈLES ML ===
              Model  ROC-AUC  F1-Score
Logistic Regression   0.6399    0.6461
  Gradient Boosting   0.6349    0.6554
      Random Forest   0.6018    0.6285


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# 1. Chargement des données
df = pd.read_csv('../data/processed_mro_ml.csv')

# 2. X/y Split & One-Hot Encoding
X = df.drop(columns=['po_id', 'is_late'])
y = df['is_late']

# Encoding des catégorielles (site_id, supplier_id, part_family)
X_encoded = pd.get_dummies(X, drop_first=True)

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Modèle Gradient Boosting
model = GradientBoostingClassifier(
    n_estimators=150, 
    learning_rate=0.08, 
    max_depth=4, 
    random_state=42
)
model.fit(X_train, y_train)

# 5. Évaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== PERFORMANCES GRADIENT BOOSTING ===")
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred))

# 6. Exporter le bundle (modèle + colonnes exactes d'entraînement)
model_bundle = {
    'model': model,
    'features': list(X_encoded.columns)
}

joblib.dump(model_bundle, '../models/mro_risk_model.pkl')
print("Modèle exporté avec succès dans '../models/mro_risk_model.pkl'")

=== PERFORMANCES DU GRADIENT BOOSTING ===
ROC-AUC Score : 0.6349

              precision    recall  f1-score   support

           0       0.57      0.59      0.58      2620
           1       0.66      0.65      0.66      3314

    accuracy                           0.62      5934
   macro avg       0.62      0.62      0.62      5934
weighted avg       0.62      0.62      0.62      5934


Modèle Gradient Boosting exporté avec succès dans '../models/mro_risk_model.pkl' !
